# Healthcare Analytics Demo — One-Click Launcher

**Run All** to deploy the complete Healthcare Payer/Provider analytics solution into this Fabric workspace.

### What Gets Deployed
| Layer | Items |
|-------|-------|
| **Lakehouses** | `lh_bronze_raw`, `lh_silver_stage`, `lh_silver_ods`, `lh_gold_curated` |
| **Notebooks** | 5 ETL + 2 data generation + 5 RTI (event simulator, Eventhouse setup, 3 scoring) |
| **Pipelines** | `PL_Healthcare_Full_Load`, `PL_Healthcare_Master` |
| **Semantic Model** | `HealthcareDemoHLS` (star schema) |
| **Data Agent** | `HealthcareHLSAgent` (Copilot AI) |
| **Ontology + Graph** | `Healthcare_Demo_Ontology_HLS` + `Healthcare_Demo_Graph` (deployed via API after pipeline) |
| **Real-Time Intelligence** | Eventhouse + KQL DB + 3 scoring use cases (fraud, care gaps, high-cost trajectory) |

### Prerequisites
- An **empty** Fabric workspace (this notebook should be the only item)
- Fabric capacity: **F64** or higher recommended
- User must be workspace **Admin** or **Member**

### Configuration
Edit the cell below to point to your GitHub repo (public or private).

In [ ]:
# ============================================================================
# CELL 1 — Install fabric-launcher
# ============================================================================

%pip install fabric-launcher --quiet

import notebookutils
from fabric_launcher import FabricLauncher

print("fabric-launcher installed successfully")

In [ ]:
# ============================================================================
# CONFIGURATION — Edit these values
# ============================================================================

GITHUB_OWNER = "rasgiza"                # GitHub org or user (public mirror)
GITHUB_REPO  = "Fabric-Payer-Provider-HealthCare-Demo"
GITHUB_BRANCH = "main"
GITHUB_TOKEN = ""                    # Leave empty for public repos. Only needed for private repos (classic PAT with repo scope).

# Deployment options
GENERATE_DATA = True                  # Generate fresh synthetic data
RUN_PIPELINE = True                   # Run the full-load pipeline after data gen
UPLOAD_KNOWLEDGE_DOCS = True          # Upload healthcare knowledge docs to lakehouse
DEPLOY_RTI = True                     # Deploy Real-Time Intelligence (Eventhouse + scoring)

print(f"Will deploy from: github.com/{GITHUB_OWNER}/{GITHUB_REPO} @ {GITHUB_BRANCH}")

In [ ]:
# ============================================================================
# CELL 2 — Initialize launcher
# ============================================================================

launcher = FabricLauncher(
    notebookutils,
    environment="DEV",
    allow_non_empty_workspace=False,
    fix_zero_logical_ids=True,
    debug=False,
)

workspace_id = notebookutils.runtime.context.get("currentWorkspaceId", "")
print(f"Workspace ID: {workspace_id}")
print(f"Launcher ready")

In [ ]:
# ============================================================================
# CELL 3 — Download repo & deploy all Fabric artifacts from scratch (staged)
# ============================================================================
# Downloads the repo as a ZIP, extracts it, and creates every artifact
# in the workspace using the Fabric REST API.  Order matters because
# later items reference earlier ones via logicalId.
#
# Stage 1: Lakehouses          — must exist before notebooks reference them
# Stage 2: Eventhouse           — async provisioning, needs its own stage
# Stage 3: KQL Database         — depends on Eventhouse being fully ready
# Stage 4: Notebooks (create)  — creates notebook items in workspace
# Stage 5: Notebooks (content) — updateDefinition applies code reliably
#           (Fabric Create Item API can silently drop inline definitions
#            for notebooks; a second pass forces updateDefinition which
#            always applies the content.)
# Stage 6: Data Pipelines      — orchestrate notebook execution
# Stage 7: Data Agent     — SM created later (Cell 9) after Gold tables exist
# ============================================================================

print("Downloading repo and deploying artifacts...")
print("This takes 3-5 minutes.\n")

downloader, deployer, report = launcher.download_and_deploy(
    repo_owner=GITHUB_OWNER,
    repo_name=GITHUB_REPO,
    branch=GITHUB_BRANCH,
    github_token=GITHUB_TOKEN if GITHUB_TOKEN else None,
    workspace_folder="workspace",
    item_type_stages=[
        ["Lakehouse"],                         # Stage 1
        ["Eventhouse"],                        # Stage 2 — provision Eventhouse first
        ["KQLDatabase"],                       # Stage 3 — KQL DB needs Eventhouse ready
        ["Notebook"],                          # Stage 4 — create notebook items
        ["Notebook"],                          # Stage 5 — updateDefinition ensures code is applied
        ["DataPipeline"],                      # Stage 6
        ["DataAgent"],                          # Stage 7 (SM created in Cell 9 after pipeline)
    ],
    validate_after_deployment=True,
    generate_report=True,
    deployment_retries=2,
)

print("\n" + "=" * 60)
print("  ARTIFACT DEPLOYMENT COMPLETE")
print("=" * 60)

In [ ]:
# ============================================================================
# CELL 3b -- Push notebook content via ipynb format (guaranteed to work)
# ============================================================================
# The Fabric Create Item API silently creates empty notebook shells.
# The generic updateDefinition with .py format returns 200 but doesn't apply.
# Solution: Convert .py -> .ipynb format and push via the notebook-specific
# updateDefinition API, which documents ipynb as the supported format.
# ============================================================================

import base64, requests, os, json, time, re

print("Converting notebooks to ipynb and pushing via REST API...\n")

token = notebookutils.credentials.getToken("pbi")
hdrs = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
api = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}"

# -- Helper: Convert Fabric .py source -> ipynb JSON ----------------------
def py_to_ipynb(py_content, metadata_json=None, logical_to_real=None, ws_id=None):
    """Convert Fabric notebook .py source to Jupyter ipynb JSON."""
    # Replace %pip magic with subprocess (disabled in child notebooks via notebookutils.notebook.run)
    # Also purge cached azure.* modules so re-import picks up newly installed versions
    py_content = re.sub(
        r'^%pip install (.+)$',
        r'import subprocess, sys; subprocess.check_call([sys.executable, "-m", "pip", "install"] + "\1".split()); [sys.modules.pop(k) for k in sorted(sys.modules) if k.startswith("azure")]',
        py_content,
        flags=re.MULTILINE,
    )
    lines = py_content.split("\n")
    cells = []
    i = 0

    # Skip header
    while i < len(lines) and (lines[i].strip() in ("", "# Fabric notebook source")):
        i += 1

    while i < len(lines):
        # Look for METADATA marker
        if lines[i].startswith("# METADATA **"):
            i += 1
            while i < len(lines) and lines[i].strip() == "":
                i += 1
            if i >= len(lines):
                break

            # Determine cell type
            if lines[i].startswith("# CELL **"):
                cell_type = "code"
            elif lines[i].startswith("# MARKDOWN **"):
                cell_type = "markdown"
            else:
                i += 1
                continue

            i += 1
            # Skip blank line after marker
            if i < len(lines) and lines[i].strip() == "":
                i += 1

            # Collect content until next METADATA or EOF
            cell_lines = []
            while i < len(lines) and not lines[i].startswith("# METADATA **"):
                cell_lines.append(lines[i])
                i += 1

            # Remove trailing blank lines
            while cell_lines and cell_lines[-1].strip() == "":
                cell_lines.pop()

            # For markdown: remove "# " prefix
            if cell_type == "markdown":
                processed = []
                for ln in cell_lines:
                    if ln.startswith("# "):
                        processed.append(ln[2:])
                    elif ln == "#":
                        processed.append("")
                    else:
                        processed.append(ln)
                cell_lines = processed

            if cell_lines:
                # Build source array (each line gets \n except the last)
                source = [ln + "\n" for ln in cell_lines[:-1]]
                source.append(cell_lines[-1])

                nb_cell = {"cell_type": cell_type, "metadata": {}, "source": source}
                if cell_type == "code":
                    nb_cell["outputs"] = []
                    nb_cell["execution_count"] = None
                cells.append(nb_cell)
        else:
            i += 1

    # Build notebook structure
    nb_metadata = {
        "kernel_info": {"name": "synapse_pyspark"},
        "kernelspec": {
            "name": "synapse_pyspark",
            "display_name": "Synapse PySpark",
            "language": "Python",
        },
        "language_info": {"name": "python"},
    }

    # Merge lakehouse dependencies from metadata JSON
    if metadata_json:
        deps = metadata_json.get("dependencies", {})
        if deps:
            # Replace logical IDs in the metadata
            deps_str = json.dumps(deps)
            if logical_to_real:
                for lid, rid in logical_to_real.items():
                    deps_str = deps_str.replace(lid, rid)
            if ws_id:
                deps_str = deps_str.replace("00000000-0000-0000-0000-000000000000", ws_id)
            nb_metadata["dependencies"] = json.loads(deps_str)

    return json.dumps(
        {"nbformat": 4, "nbformat_minor": 5, "metadata": nb_metadata, "cells": cells},
        indent=1,
    )

# -- Step 1: Get all deployed items --------------------------------------
resp = requests.get(f"{api}/items", headers=hdrs)
all_items = resp.json().get("value", [])
name_to_id = {(it["type"], it["displayName"]): it["id"] for it in all_items}

# -- Step 2: Locate extracted workspace directory ------------------------
ws_dir = None
for base in [".lakehouse/default/Files/src", "/lakehouse/default/Files/src"]:
    # Direct path
    d = os.path.join(base, "workspace")
    if os.path.isdir(d):
        ws_dir = d
        break
    # One level deep (fabric-launcher extracts ZIP to a subdirectory)
    if os.path.isdir(base):
        for sub in os.listdir(base):
            d = os.path.join(base, sub, "workspace")
            if os.path.isdir(d):
                ws_dir = d
                break
    if ws_dir:
        break

if not ws_dir:
    print("ERROR: Could not find extracted workspace directory.")
    print("  Searched .lakehouse/default/Files/src/ (direct and one level deep)")
    if os.path.isdir(".lakehouse/default/Files/src"):
        print("  Contents of Files/src/:", os.listdir(".lakehouse/default/Files/src"))
    elif os.path.isdir("/lakehouse/default/Files/src"):
        print("  Contents of Files/src/:", os.listdir("/lakehouse/default/Files/src"))
else:
    # Build logical_id -> real_id mapping
    logical_to_real = {}
    for entry in os.listdir(ws_dir):
        pf = os.path.join(ws_dir, entry, ".platform")
        if os.path.exists(pf):
            with open(pf, "r") as f:
                plat = json.load(f)
            itype = plat["metadata"]["type"]
            iname = plat["metadata"]["displayName"]
            lid = plat["config"]["logicalId"]
            rid = name_to_id.get((itype, iname), "")
            if lid and rid:
                logical_to_real[lid] = rid

    # -- Step 3: Convert and push each notebook --------------------------
    pushed, failed = 0, 0
    for entry in sorted(os.listdir(ws_dir)):
        if not entry.endswith(".Notebook"):
            continue
        nb_name = entry.replace(".Notebook", "")
        nb_id = name_to_id.get(("Notebook", nb_name))
        if not nb_id:
            print(f"  SKIP {nb_name}: not in workspace")
            continue

        nb_path = os.path.join(ws_dir, entry)

        # Read .py content
        py_file = os.path.join(nb_path, "notebook-content.py")
        if not os.path.exists(py_file):
            print(f"  SKIP {nb_name}: no notebook-content.py")
            continue
        with open(py_file, "r", encoding="utf-8") as f:
            py_content = f.read()

        # Read metadata JSON
        meta_file = os.path.join(nb_path, ".metadata", "notebook-metadata.json")
        meta_json = None
        if os.path.exists(meta_file):
            with open(meta_file, "r", encoding="utf-8") as f:
                meta_json = json.load(f)

        # Convert to ipynb
        ipynb_content = py_to_ipynb(py_content, meta_json, logical_to_real, workspace_id)
        ipynb_b64 = base64.b64encode(ipynb_content.encode("utf-8")).decode("utf-8")

        # Build payload -- ipynb file + .platform (required for updateMetadata=true)
        platform_obj = {
            "$schema": "https://developer.microsoft.com/json-schemas/fabric/gitIntegration/platformProperties/2.0.0/schema.json",
            "metadata": {"type": "Notebook", "displayName": nb_name},
            "config": {"version": "2.0", "logicalId": nb_id},
        }
        platform_b64 = base64.b64encode(json.dumps(platform_obj, indent=2).encode("utf-8")).decode("utf-8")
        parts = [
            {"path": "notebook-content.ipynb", "payload": ipynb_b64, "payloadType": "InlineBase64"},
            {"path": ".platform", "payload": platform_b64, "payloadType": "InlineBase64"},
        ]
        body = {"definition": {"format": "ipynb", "parts": parts}}

        # POST to updateDefinition (with retry for transient 500s)
        r = None
        for attempt in range(3):
            r = requests.post(
                f"{api}/items/{nb_id}/updateDefinition?updateMetadata=true",
                headers=hdrs,
                json=body,
            )
            if r.status_code in (200, 202):
                break
            if r.status_code >= 500 and attempt < 2:
                print(f"  {nb_name}: HTTP {r.status_code}, retrying ({attempt+1}/3)...")
                time.sleep(5 * (attempt + 1))
            else:
                break

        # Handle LRO (202)
        if r.status_code == 202:
            loc = r.headers.get("Location", "")
            retry_after = int(r.headers.get("Retry-After", "3"))
            if loc:
                for _ in range(60):
                    time.sleep(retry_after)
                    pr = requests.get(loc, headers=hdrs)
                    if pr.status_code == 200:
                        break
                    elif pr.status_code != 202:
                        break

        if r.status_code in (200, 202):
            # Count cells for feedback
            cell_count = ipynb_content.count('"cell_type"')
            size_kb = round(len(ipynb_content) / 1024, 1)
            pushed += 1
            print(f"  {nb_name}: OK ({cell_count} cells, {size_kb}KB)")
        else:
            failed += 1
            print(f"  FAIL {nb_name}: HTTP {r.status_code} -- {r.text[:300]}")

    print(f"\n{'='*60}")
    print(f"  Notebooks pushed: {pushed} (ipynb format)")
    if failed:
        print(f"  Failed: {failed}")

    # -- Step 4: Verify --------------------------------------------------
    print(f"\nVerifying content (getDefinition on 01_Bronze_Ingest_CSV)...")
    test_id = name_to_id.get(("Notebook", "01_Bronze_Ingest_CSV"))
    if test_id:
        vr = requests.post(f"{api}/items/{test_id}/getDefinition", headers=hdrs, json={})
        # Handle LRO for getDefinition
        if vr.status_code == 202:
            loc = vr.headers.get("Location", "")
            if loc:
                for _ in range(30):
                    time.sleep(2)
                    vr = requests.get(loc, headers=hdrs)
                    if vr.status_code == 200:
                        break

        if vr.status_code == 200:
            vdata = vr.json()
            defn_parts = vdata.get("definition", {}).get("parts", [])
            print(f"  Parts returned: {[p.get('path') for p in defn_parts]}")
            for p in defn_parts:
                if "content" in p.get("path", "").lower():
                    decoded = base64.b64decode(p["payload"]).decode("utf-8")
                    has_spark = "spark" in decoded.lower() or "bronze" in decoded.lower()
                    size = len(decoded)
                    print(f"  Content size: {size:,} bytes, has_code: {has_spark}")
                    if size > 500:
                        print(f"  [OK] Notebook content verified -- has real code")
                    else:
                        print(f"  [!] Content is suspiciously small: {decoded[:200]}")
                    break
            else:
                print(f"  [!] No content part found. All parts: {json.dumps([{k:v for k,v in p.items() if k != 'payload'} for p in defn_parts], indent=2)}")
        else:
            print(f"  getDefinition returned HTTP {vr.status_code}: {vr.text[:200]}")
    print(f"{'='*60}")

In [ ]:
# ============================================================================
# CELL 4 — Upload healthcare knowledge docs to Data Agent lakehouse
# ============================================================================

if UPLOAD_KNOWLEDGE_DOCS:
    import os, glob

    # fabric-launcher extracts the repo ZIP to this relative path by default
    # Try both relative (library default) and absolute (Fabric mount) paths
    for candidate in [".lakehouse/default/Files/src", "/lakehouse/default/Files/src"]:
        if os.path.isdir(candidate):
            extract_base = candidate
            break
    else:
        print("Warning: Could not find extract directory. Skipping knowledge doc upload.")
        print("  Tried: .lakehouse/default/Files/src and /lakehouse/default/Files/src")
        UPLOAD_KNOWLEDGE_DOCS = False
        extract_base = None

    # fabric-launcher extracts repo contents directly into extract_base (no wrapper folder)
    repo_root = extract_base

    knowledge_src = os.path.join(repo_root, "healthcare_knowledge") if repo_root else None

    if knowledge_src and os.path.exists(knowledge_src):
        print("Uploading healthcare knowledge documents...")
        launcher.upload_files_to_lakehouse(
            lakehouse_name="lh_gold_curated",
            source_directory=knowledge_src,
            target_folder="healthcare_knowledge",
            file_patterns=["*.md"],
        )
        # Count uploaded files
        count = sum(len(files) for _, _, files in os.walk(knowledge_src) if files)
        print(f"Uploaded {count} knowledge documents to lh_gold_curated/Files/healthcare_knowledge/")
    else:
        print(f"Warning: healthcare_knowledge folder not found at {knowledge_src}")
else:
    print("Skipping knowledge doc upload (UPLOAD_KNOWLEDGE_DOCS=False)")

In [ ]:
# ============================================================================
# CELL 5 — Generate synthetic data
# ============================================================================
# Uses notebookutils.notebook.run() — the native Fabric way to orchestrate
# notebooks. More reliable than the Jobs REST API after updateDefinition.
# ============================================================================

if GENERATE_DATA:
    print("Running NB_Generate_Sample_Data (generates ~10K patients, 100K encounters)...")
    print("This takes 2-4 minutes.\n")

    try:
        notebookutils.notebook.run("NB_Generate_Sample_Data", 1200, {"useRootDefaultLakehouse": True})
        print("\n✅ Data generation SUCCEEDED")
    except Exception as e:
        print(f"\n❌ Data generation FAILED: {e}")
        print("Try running NB_Generate_Sample_Data manually from the workspace.")
else:
    print("Skipping data generation (GENERATE_DATA=False)")

In [ ]:
# ============================================================================
# CELL 6 — Run the full-load pipeline
# ============================================================================

if RUN_PIPELINE:
    import requests, time, json

    token = notebookutils.credentials.getToken("pbi")
    headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
    api_base = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}"

    # Find the pipeline ID
    print("Looking up PL_Healthcare_Master pipeline...")
    resp = requests.get(f"{api_base}/items?type=DataPipeline", headers=headers)
    resp.raise_for_status()
    pipelines = resp.json().get("value", [])
    pipeline = next((p for p in pipelines if p["displayName"] == "PL_Healthcare_Master"), None)

    if not pipeline:
        print("WARNING: PL_Healthcare_Master not found. Skipping pipeline run.")
        print("You can run it manually from the workspace.")
    else:
        pipeline_id = pipeline["id"]
        print(f"Pipeline ID: {pipeline_id}")

        # Trigger pipeline with load_mode=full
        print("Triggering pipeline with load_mode=full...")
        trigger_body = {
            "executionData": {
                "parameters": {
                    "load_mode": "full"
                }
            }
        }
        resp = requests.post(
            f"{api_base}/items/{pipeline_id}/jobs/instances?jobType=Pipeline",
            headers=headers,
            json=trigger_body,
        )

        if resp.status_code in (200, 202):
            # Get job ID from Location header or response
            location = resp.headers.get("Location", "")
            print(f"Pipeline triggered. Polling for completion...")
            print(f"(This takes 8-15 minutes for full load)\n")

            # Poll until complete
            max_polls = 120  # 120 * 15s = 30 min max
            for poll in range(max_polls):
                time.sleep(15)
                try:
                    if location:
                        poll_resp = requests.get(location, headers=headers)
                    else:
                        poll_resp = requests.get(
                            f"{api_base}/items/{pipeline_id}/jobs/instances",
                            headers=headers,
                        )
                    if poll_resp.status_code == 200:
                        job_data = poll_resp.json()
                        status = job_data.get("status", "Unknown")
                        if status in ("Completed", "Succeeded"):
                            print(f"  Pipeline COMPLETED after {(poll+1)*15}s")
                            break
                        elif status in ("Failed", "Cancelled"):
                            print(f"  Pipeline {status} after {(poll+1)*15}s")
                            print(f"  Check the pipeline run history for details.")
                            break
                        else:
                            if poll % 4 == 0:  # Print every 60s
                                print(f"  [{(poll+1)*15}s] Status: {status}...")
                except Exception as e:
                    if poll % 4 == 0:
                        print(f"  [{(poll+1)*15}s] Polling... ({e})")
            else:
                print("  Pipeline still running after 30 min. Check workspace for status.")
        else:
            print(f"  Pipeline trigger returned {resp.status_code}: {resp.text}")
            print("  You can run it manually from the workspace.")
else:
    print("Skipping pipeline run (RUN_PIPELINE=False)")
    print("To run manually: Open PL_Healthcare_Master → Run with load_mode=full")

In [ ]:
# ============================================================================
# CELL 6b -- Create & Refresh Semantic Model (Direct Lake)
# ============================================================================
# The SM is NOT deployed via fabric-launcher (placeholder GUIDs in the repo
# would corrupt AS internal metadata). Instead we create it here, AFTER the
# pipeline has run and Gold tables exist in lh_gold_curated.
#
# Steps:
#   1. Find lh_gold_curated lakehouse ID
#   2. Delete any existing SM with the same name (idempotent re-run)
#   3. Load all TMDL files from the extracted repo on the lakehouse filesystem
#   4. Patch expressions.tmdl URL with correct workspace/lakehouse IDs
#   5. Create fresh SM via POST /semanticModels with all definition parts
#   6. Trigger Full refresh
# ============================================================================

import requests, time, base64, re, json, os

token = notebookutils.credentials.getToken("pbi")
headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
FABRIC_API = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}"

SM_NAME = "HealthcareDemoHLS"
SM_REPO_DIR = "workspace/HealthcareDemoHLS.SemanticModel"
URL_PATTERN = re.compile(
    r'https://onelake\.dfs\.fabric\.microsoft\.com/'
    r'[0-9a-fA-F-]{36}/[0-9a-fA-F-]{36}'
)

def wait_lro(resp, hdr, timeout=120):
    """Poll an LRO and return (final_status, result_body)."""
    loc = resp.headers.get("Location", "")
    retry = int(resp.headers.get("Retry-After", 5))
    elapsed = 0
    while elapsed < timeout:
        time.sleep(retry)
        elapsed += retry
        r = requests.get(loc, headers=hdr)
        if r.status_code != 200:
            continue
        body = r.json()
        st = body.get("status", "")
        if st == "Succeeded":
            res_loc = body.get("resourceLocation", "")
            if res_loc:
                rr = requests.get(res_loc, headers=hdr)
                if rr.status_code == 200:
                    return "Succeeded", rr.json()
            rr = requests.get(f"{loc}/result", headers=hdr)
            if rr.status_code == 200:
                return "Succeeded", rr.json()
            return "Succeeded", body
        elif st in ("Failed", "Cancelled"):
            err = body.get("error", {})
            return st, err
    return "Timeout", {}

print("=" * 60)
print("  SEMANTIC MODEL -- Create from Repo Definition")
print("=" * 60)

# -- Step 1: Discover lakehouse ID ----------------------------------------
resp = requests.get(f"{FABRIC_API}/lakehouses", headers=headers)
resp.raise_for_status()
lh_gold_id = None
for lh in resp.json().get("value", []):
    if lh["displayName"] == "lh_gold_curated":
        lh_gold_id = lh["id"]
        break

if not lh_gold_id:
    print("  [WARN] lh_gold_curated not found -- skipping SM creation")
    print("  Run the pipeline first, then re-run this cell.")
else:
    print(f"  Lakehouse: lh_gold_curated ({lh_gold_id})")
    print(f"  Workspace: {workspace_id}")
    new_url = f"https://onelake.dfs.fabric.microsoft.com/{workspace_id}/{lh_gold_id}"

    # -- Step 2: Delete existing SM if any (idempotent re-run) -------------
    resp = requests.get(f"{FABRIC_API}/items?type=SemanticModel", headers=headers)
    resp.raise_for_status()
    for sm in resp.json().get("value", []):
        if sm["displayName"] == SM_NAME:
            old_id = sm["id"]
            print(f"  Deleting existing SM: {SM_NAME} ({old_id})...")
            dr = requests.delete(f"{FABRIC_API}/semanticModels/{old_id}", headers=headers)
            if dr.status_code in (200, 202, 204):
                if dr.status_code == 202:
                    wait_lro(dr, headers, timeout=60)
                print(f"  Deleted.")
            else:
                print(f"  [WARN] Delete HTTP {dr.status_code}: {dr.text[:200]}")
            print("  Waiting 10s for name availability...")
            time.sleep(10)
            break

    # -- Step 3: Load TMDL from extracted repo on lakehouse ----------------
    # fabric-launcher extracts to .lakehouse/default/Files/src/<repodir>/
    sm_base = None
    for base_prefix in [".lakehouse/default/Files/src", "/lakehouse/default/Files/src"]:
        if not os.path.isdir(base_prefix):
            continue
        # Direct path
        direct = os.path.join(base_prefix, SM_REPO_DIR)
        if os.path.isdir(direct):
            sm_base = direct
            break
        # One level down (extracted repo subdirectory)
        for sub in os.listdir(base_prefix):
            nested = os.path.join(base_prefix, sub, SM_REPO_DIR)
            if os.path.isdir(nested):
                sm_base = nested
                break
        if sm_base:
            break

    if not sm_base:
        print("  [FAIL] SM definition directory not found on lakehouse filesystem.")
        print("  Expected: Files/src/[<repo-dir>/]workspace/HealthcareDemoHLS.SemanticModel/")
        # List what's actually there for debugging
        for base_prefix in [".lakehouse/default/Files/src", "/lakehouse/default/Files/src"]:
            if os.path.isdir(base_prefix):
                print(f"  Contents of {base_prefix}:")
                for item in os.listdir(base_prefix)[:10]:
                    print(f"    {item}")
    else:
        print(f"  SM definition dir: {sm_base}")
        def_dir = os.path.join(sm_base, "definition")

        parts = []

        # Load definition.pbism
        pbism_path = os.path.join(sm_base, "definition.pbism")
        if os.path.exists(pbism_path):
            with open(pbism_path, "rb") as f:
                raw = f.read()
            if raw.startswith(b"\xef\xbb\xbf"):
                raw = raw[3:]
            parts.append({
                "path": "definition.pbism",
                "payload": base64.b64encode(raw).decode(),
                "payloadType": "InlineBase64"
            })

        # Load all definition/*.tmdl files (recursively for tables/)
        if os.path.isdir(def_dir):
            for root, dirs, files in os.walk(def_dir):
                for fname in sorted(files):
                    fpath = os.path.join(root, fname)
                    rel = "definition/" + os.path.relpath(fpath, def_dir).replace("\\", "/")
                    with open(fpath, "rb") as f:
                        raw = f.read()
                    if raw.startswith(b"\xef\xbb\xbf"):
                        raw = raw[3:]

                    # Patch expressions.tmdl URL
                    if "expressions.tmdl" in rel:
                        content = raw.decode("utf-8")
                        matches = URL_PATTERN.findall(content)
                        if matches and matches[0] != new_url:
                            print(f"  Patching URL: {matches[0]} -> {new_url}")
                            content = URL_PATTERN.sub(new_url, content)
                            raw = content.encode("utf-8")
                        elif matches:
                            print(f"  URL already correct in expressions.tmdl")
                        else:
                            print(f"  [WARN] No URL found in expressions.tmdl -- injecting correct URL")

                    parts.append({
                        "path": rel,
                        "payload": base64.b64encode(raw).decode(),
                        "payloadType": "InlineBase64"
                    })

        print(f"  Loaded {len(parts)} definition parts:")
        for p in parts[:5]:
            print(f"    {p['path']}")
        if len(parts) > 5:
            print(f"    ... and {len(parts) - 5} more")

        if not parts:
            print("  [FAIL] No definition parts loaded. Cannot create SM.")
        else:
            # -- Step 4: Create SM with correct definition -----------------
            token = notebookutils.credentials.getToken("pbi")
            headers["Authorization"] = f"Bearer {token}"

            create_body = {
                "displayName": SM_NAME,
                "description": "Healthcare Demo Direct Lake semantic model",
                "definition": {"parts": parts}
            }

            print(f"  Creating SM: {SM_NAME} with {len(parts)} TMDL parts...")
            r = requests.post(f"{FABRIC_API}/semanticModels", headers=headers, json=create_body)
            print(f"  Create HTTP {r.status_code}")

            sm_id = None
            if r.status_code in (200, 201):
                sm_id = r.json().get("id")
                print(f"  Created: {sm_id}")
            elif r.status_code == 202:
                st, body = wait_lro(r, headers, timeout=180)
                if st == "Succeeded":
                    time.sleep(3)
                    resp2 = requests.get(f"{FABRIC_API}/items?type=SemanticModel", headers=headers)
                    for sm in resp2.json().get("value", []):
                        if sm["displayName"] == SM_NAME:
                            sm_id = sm["id"]
                            break
                if sm_id:
                    print(f"  Created (async): {sm_id}")
                else:
                    print(f"  Create LRO {st}: {body}")
            else:
                print(f"  [FAIL] Create SM: HTTP {r.status_code}")
                print(f"  Response: {r.text[:500]}")
                sm_id = None

            # -- Step 5: Trigger refresh -----------------------------------
            if sm_id:
                print("  Waiting 15s for SM initialization...")
                time.sleep(15)

                token = notebookutils.credentials.getToken("pbi")
                headers["Authorization"] = f"Bearer {token}"

                pbi_base = f"https://api.powerbi.com/v1.0/myorg/groups/{workspace_id}"
                refresh_url = f"{pbi_base}/datasets/{sm_id}/refreshes"

                MAX_ATTEMPTS = 3
                for attempt in range(1, MAX_ATTEMPTS + 1):
                    print(f"  Triggering refresh (attempt {attempt}/{MAX_ATTEMPTS})...")
                    r = requests.post(refresh_url, headers=headers, json={"type": "Full"})

                    if r.status_code not in (200, 202):
                        print(f"  Refresh trigger HTTP {r.status_code}: {r.text[:300]}")
                        if attempt < MAX_ATTEMPTS:
                            print(f"  Retrying in 15s...")
                            time.sleep(15)
                            token = notebookutils.credentials.getToken("pbi")
                            headers["Authorization"] = f"Bearer {token}"
                            continue
                        else:
                            print("  All attempts failed. Refresh manually from the workspace.")
                            break

                    # Poll for completion
                    success = False
                    for poll_i in range(60):
                        time.sleep(10)
                        poll_resp = requests.get(refresh_url, headers=headers)
                        if poll_resp.status_code == 200:
                            refreshes = poll_resp.json().get("value", [])
                            if refreshes:
                                latest = refreshes[0]
                                status = latest.get("status", "Unknown")
                                if status in ("Completed", "Succeeded"):
                                    print(f"  Refresh COMPLETED after {(poll_i+1)*10}s")
                                    success = True
                                    break
                                elif status == "Failed":
                                    err_msg = latest.get("serviceExceptionJson", "")
                                    print(f"  Refresh FAILED after {(poll_i+1)*10}s")
                                    if err_msg:
                                        try:
                                            err_obj = json.loads(err_msg)
                                            print(f"  Error: {err_obj.get('errorCode', '')}")
                                            print(f"  Detail: {err_obj.get('errorDescription', '')[:300]}")
                                        except Exception:
                                            print(f"  Error: {err_msg[:300]}")
                                    break
                            elif poll_i % 3 == 0:
                                print(f"  [{(poll_i+1)*10}s] Status: {status}...")
                    else:
                        print("  Refresh still running after 10 min. Check workspace.")
                        break

                    if success:
                        break
                    elif attempt < MAX_ATTEMPTS:
                        print(f"  Retrying in 20s...")
                        time.sleep(20)
                        token = notebookutils.credentials.getToken("pbi")
                        headers["Authorization"] = f"Bearer {token}"
                    else:
                        print("  All refresh attempts failed. Refresh manually from the workspace.")

print("=" * 60)


In [ ]:
# ============================================================================
# CELL 10a -- Deploy Ontology via Inline API Calls
# ============================================================================
# Deploys the ontology (10 entity types, 14 relationships) directly via
# the Fabric REST API. Graph Model is auto-provisioned by Fabric.
# For manual graph deploy/diagnostics, use NB_Deploy_Graph_Model notebook.
# ============================================================================
#
# ============================================================================

import json, requests, time, base64, os, re, math, uuid

print("=" * 60)
print("  ONTOLOGY -- API Deployment")
print("=" * 60)
print()

# -- Auth & Discovery -----------------------------------------
token = notebookutils.credentials.getToken("pbi")
headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
API = "https://api.fabric.microsoft.com/v1"

print(f"  Workspace: {workspace_id}")

# Discover lh_gold_curated lakehouse ID
resp = requests.get(f"{API}/workspaces/{workspace_id}/lakehouses", headers=headers)
resp.raise_for_status()
lh_gold_id = None
for lh in resp.json().get("value", []):
    if lh["displayName"] == "lh_gold_curated":
        lh_gold_id = lh["id"]
        break
if not lh_gold_id:
    print("  [FAIL] lh_gold_curated not found -- run pipeline first")
    raise RuntimeError("lh_gold_curated not found")
print(f"  Lakehouse: lh_gold_curated ({lh_gold_id})")

# -- Paths -----------------------------------------------------
ONTOLOGY_NAME = "Healthcare_Demo_Ontology_HLS"
GRAPH_MODEL_NAME = "Healthcare_Demo_Graph"
ONT_API = f"{API}/workspaces/{workspace_id}/ontologies"
GM_API = f"{API}/workspaces/{workspace_id}/graphModels"

# Find the extracted repo -- launcher extracts ZIP to a subdirectory
# e.g. .lakehouse/default/Files/src/Fabric-Payer-Provider-HealthCare-Demo-main/
ont_dir = None
for base in [".lakehouse/default/Files/src", "/lakehouse/default/Files/src"]:
    if not os.path.isdir(base):
        continue
    # Check direct: base/ontology/ONTOLOGY_NAME
    direct = os.path.join(base, "ontology", ONTOLOGY_NAME)
    if os.path.isdir(direct):
        ont_dir = direct
        break
    # Check one level down (extracted repo subdirectory)
    for sub in os.listdir(base):
        nested = os.path.join(base, sub, "ontology", ONTOLOGY_NAME)
        if os.path.isdir(nested):
            ont_dir = nested
            break
    if ont_dir:
        break

if not ont_dir:
    print("  [WARN] Ontology dir not found on lakehouse filesystem")
    print(f"  CWD: {os.getcwd()}")
    for p in [".lakehouse", "/lakehouse", ".lakehouse/default", "/lakehouse/default"]:
        print(f"    {p} exists: {os.path.exists(p)}")
    for base in [".lakehouse/default/Files/src", "/lakehouse/default/Files/src"]:
        if os.path.isdir(base):
            print(f"  Contents of {base}: {os.listdir(base)[:10]}")
    # Fallback: download ontology files from GitHub to a temp directory
    import tempfile
    print(f"  Downloading ontology from GitHub: {GITHUB_OWNER}/{GITHUB_REPO}@{GITHUB_BRANCH} ...")
    raw_base = f"https://raw.githubusercontent.com/{GITHUB_OWNER}/{GITHUB_REPO}/{GITHUB_BRANCH}/ontology/{ONTOLOGY_NAME}"
    r_manifest = requests.get(f"{raw_base}/manifest.json")
    r_manifest.raise_for_status()
    manifest = r_manifest.json()
    ont_dir = tempfile.mkdtemp(prefix="ontology_")
    with open(os.path.join(ont_dir, "manifest.json"), "w", encoding="utf-8") as mf:
        json.dump(manifest, mf, indent=2)
    dl_count = 0
    for part_info in manifest.get("exportedParts", []):
        part_path = part_info["path"]
        r_part = requests.get(f"{raw_base}/{part_path}")
        r_part.raise_for_status()
        dest = os.path.join(ont_dir, part_path.replace("/", os.sep))
        os.makedirs(os.path.dirname(dest), exist_ok=True)
        with open(dest, "wb") as pf:
            pf.write(r_part.content)
        dl_count += 1
    print(f"  Downloaded {dl_count} files to temp dir: {ont_dir}")
else:
    print(f"  Ontology dir: {ont_dir}")

# -- Helper: LRO wait -----------------------------------------
def wait_lro(response, label, timeout=180):
    loc = response.headers.get("Location")
    if not loc:
        time.sleep(10)
        return True
    start = time.time()
    retry = int(response.headers.get("Retry-After", 5))
    while time.time() - start < timeout:
        time.sleep(retry)
        r = requests.get(loc, headers=headers)
        if r.status_code == 200:
            body = r.json()
            status = body.get("status", "")
            if status == "Succeeded":
                return True
            if status in ("Failed", "Cancelled"):
                err = body.get("error", {})
                err_code = err.get("code", "")
                err_msg = err.get("message", "")
                print(f"    [FAIL] {label}: {status}")
                if err_code or err_msg:
                    print(f"    Error: {err_code} - {err_msg[:500]}")
                err_details = err.get("details", [])
                if err_details:
                    print(f"    Details: {json.dumps(err_details, indent=2)[:1000]}")
                if not err_code and not err_msg and not err_details:
                    print(f"    Response: {json.dumps(body, indent=2)[:500]}")
                return False
    print(f"    [FAIL] {label}: timed out after {timeout}s")
    return False

# -- Helper: Load ontology parts from disk ---------------------
def load_ontology_parts(base_path):
    parts = []
    manifest_path = os.path.join(base_path, "manifest.json")
    if os.path.exists(manifest_path):
        with open(manifest_path, 'r', encoding='utf-8-sig') as f:
            manifest = json.load(f)
        for part_info in manifest.get("exportedParts", []):
            part_path = part_info["path"]
            file_path = os.path.join(base_path, part_path)
            if not os.path.exists(file_path):
                continue
            with open(file_path, 'rb') as f:
                raw = f.read()
            if raw.startswith(b'\xef\xbb\xbf'):
                raw = raw[3:]
            parts.append({"path": part_path, "payload": base64.b64encode(raw).decode("utf-8"), "payloadType": "InlineBase64"})
    else:
        for root, _dirs, files in sorted(os.walk(base_path)):
            for fname in sorted(files):
                if fname in (".platform", "manifest.json"):
                    continue
                filepath = os.path.join(root, fname)
                rel_path = os.path.relpath(filepath, base_path).replace("\\", "/")
                with open(filepath, 'rb') as f:
                    raw = f.read()
                if raw.startswith(b'\xef\xbb\xbf'):
                    raw = raw[3:]
                parts.append({"path": rel_path, "payload": base64.b64encode(raw).decode("utf-8"), "payloadType": "InlineBase64"})
    return parts

# -- Helper: Patch data binding IDs ----------------------------
def patch_bindings(parts, ws_id, lh_id):
    patched = []
    for part in parts:
        path = part["path"]
        if "DataBindings/" in path or "Contextualizations/" in path:
            try:
                content = base64.b64decode(part["payload"]).decode("utf-8")
                obj = json.loads(content)
                _patch_ids(obj, ws_id, lh_id)
                content = json.dumps(obj, indent=2, ensure_ascii=False)
                part = {**part, "payload": base64.b64encode(content.encode("utf-8")).decode("utf-8")}
            except Exception as e:
                print(f"    [WARN] Could not patch {path}: {e}")
        patched.append(part)
    return patched

def _patch_ids(obj, ws_id, lh_id):
    if isinstance(obj, dict):
        for key in list(obj.keys()):
            val = obj[key]
            lk = key.lower()
            if lk in ("workspaceid", "workspaceguid", "workspace_id"):
                obj[key] = ws_id
            elif lk in ("itemid", "lakehouseid", "artifactid", "item_id"):
                obj[key] = lh_id
            elif isinstance(val, str) and "onelake" in val.lower():
                m = re.match(r'(abfss://)([0-9a-f-]+)(@onelake[^/]*/)([0-9a-f-]+)(.*)', val, re.I)
                if m:
                    obj[key] = f"{m.group(1)}{ws_id}{m.group(3)}{lh_id}{m.group(5)}"
            elif isinstance(val, (dict, list)):
                _patch_ids(val, ws_id, lh_id)
    elif isinstance(obj, list):
        for item in obj:
            if isinstance(item, (dict, list)):
                _patch_ids(item, ws_id, lh_id)

# ==============================================================
# STEP 1: DEPLOY ONTOLOGY
# ==============================================================
print()
print("Step 1: Deploy Ontology")
print("-" * 40)

# Load parts
parts = load_ontology_parts(ont_dir)
et_count = sum(1 for p in parts if p["path"].startswith("EntityTypes/") and p["path"].endswith("/definition.json"))
rt_count = sum(1 for p in parts if p["path"].startswith("RelationshipTypes/") and p["path"].endswith("/definition.json"))
print(f"  Loaded {len(parts)} parts: {et_count} entities, {rt_count} relationships")

# Patch data bindings
parts = patch_bindings(parts, workspace_id, lh_gold_id)
print(f"  Patched data bindings for target environment")

# -- Pre-deploy Validation: verify bindings match actual table schemas ------
print()
print("  Pre-deploy validation (bindings vs table schemas)...")
_val_errors = []
_val_fixes = []

# Helper: get table columns using Delta path (no default lakehouse needed)
_table_cols_cache = {}
def _get_table_cols(table_name):
    if table_name in _table_cols_cache:
        return _table_cols_cache[table_name]
    cols = None
    # Try 1: Spark SQL (works if default lakehouse is attached)
    try:
        _df = spark.sql(f"DESCRIBE lh_gold_curated.{table_name}")
        cols = {r["col_name"] for r in _df.collect() if not r["col_name"].startswith("#")}
    except Exception:
        pass
    # Try 2: Read Delta schema via OneLake path (no default lakehouse needed)
    if cols is None:
        try:
            _path = f"abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/{lh_gold_id}/Tables/{table_name}"
            _df = spark.read.format("delta").load(_path)
            cols = {f.name for f in _df.schema.fields}
        except Exception:
            pass
    _table_cols_cache[table_name] = cols
    return cols

# Test if we can reach any table at all
_sample_table = None
for _p in parts:
    _pp = _p["path"]
    if "DataBindings/" in _pp:
        try:
            _b = json.loads(base64.b64decode(_p["payload"]).decode("utf-8"))
            _sample_table = _b.get("dataBindingConfiguration", {}).get("sourceTableProperties", {}).get("sourceTableName")
            if _sample_table:
                break
        except Exception:
            pass
_can_validate = _get_table_cols(_sample_table) is not None if _sample_table else False
if not _can_validate:
    print("  [INFO] Table schemas not accessible (no default lakehouse context)")
    print("         Skipping column validation -- ontology push will proceed")
    print("         The binding structure and property-name consistency will still be checked")

# Build lookup: decode definition.json and DataBinding parts
_entity_defs = {}   # entity_folder_id -> definition dict
_entity_bindings = {}  # entity_folder_id -> binding dict
for _p in parts:
    _pp = _p["path"]
    if _pp.startswith("EntityTypes/") and _pp.endswith("/definition.json"):
        _folder_id = _pp.split("/")[1]
        try:
            _entity_defs[_folder_id] = json.loads(base64.b64decode(_p["payload"]).decode("utf-8"))
        except Exception:
            pass
    elif "DataBindings/" in _pp and _pp.startswith("EntityTypes/"):
        _folder_id = _pp.split("/")[1]
        try:
            _entity_bindings[_folder_id] = json.loads(base64.b64decode(_p["payload"]).decode("utf-8"))
        except Exception:
            pass

# Validate each entity: property names vs binding sourceColumnNames vs actual table columns
for _folder_id, _defn in _entity_defs.items():
    _ename = _defn.get("name", _folder_id)
    _bnd = _entity_bindings.get(_folder_id)
    if not _bnd:
        continue
    _cfg = _bnd.get("dataBindingConfiguration", {})
    _src_table = (_cfg.get("sourceTableProperties", {}).get("sourceTableName"))
    if not _src_table:
        continue

    # Get actual table columns (Spark SQL or Delta path)
    _actual_cols = _get_table_cols(_src_table) if _can_validate else None

    # Build property id -> name lookup from definition
    _prop_id_to_name = {p["id"]: p["name"] for p in _defn.get("properties", [])}

    # Check each binding
    for _pb in _cfg.get("propertyBindings", []):
        _src_col = _pb["sourceColumnName"]
        _target_pid = _pb["targetPropertyId"]
        _prop_name = _prop_id_to_name.get(_target_pid, "?")

        # Check 1: sourceColumnName must exist in actual table (only if we can validate)
        if _actual_cols is not None and _src_col not in _actual_cols:
            _val_errors.append(f"  {_ename}: binding column '{_src_col}' not in {_src_table} (actual: {sorted(_actual_cols)[:8]}...)")

        # Check 2: property name should match sourceColumnName for auto-provisioned graph
        # Skip columns starting with '_' (metadata/system cols like _load_timestamp)
        if _prop_name != _src_col and _prop_name != "?" and not _src_col.startswith("_"):
            _val_fixes.append((_folder_id, _target_pid, _prop_name, _src_col, _ename))

# Auto-fix property name mismatches in parts (in memory)
if _val_fixes:
    print(f"  Found {len(_val_fixes)} property name mismatches -- auto-fixing in parts...")
    _fix_lookup = {}  # (folder_id, prop_id) -> new_name
    for _folder_id, _pid, _old, _new, _ename in _val_fixes:
        _fix_lookup[(_folder_id, _pid)] = _new
        print(f"    {_ename}: '{_old}' -> '{_new}'")

    # Patch definition.json parts in memory
    for _pi, _p in enumerate(parts):
        _pp = _p["path"]
        if not (_pp.startswith("EntityTypes/") and _pp.endswith("/definition.json")):
            continue
        _folder_id = _pp.split("/")[1]
        _needs_fix = any(fid == _folder_id for (fid, _) in _fix_lookup)
        if not _needs_fix:
            continue
        try:
            _d = json.loads(base64.b64decode(_p["payload"]).decode("utf-8"))
            _changed = False
            for _prop in _d.get("properties", []):
                _key = (_folder_id, _prop["id"])
                if _key in _fix_lookup:
                    _prop["name"] = _fix_lookup[_key]
                    _changed = True
            if _changed:
                parts[_pi] = {**_p, "payload": base64.b64encode(
                    json.dumps(_d, indent=2, ensure_ascii=False).encode("utf-8")
                ).decode("utf-8")}
        except Exception:
            pass

# Validate contextualization columns (only if tables are accessible)
_ctx_errors = []
if _can_validate:
    for _p in parts:
        _pp = _p["path"]
        if "Contextualizations/" not in _pp:
            continue
        try:
            _ctx = json.loads(base64.b64decode(_p["payload"]).decode("utf-8"))
            _ctx_table = _ctx.get("dataBindingTable", {}).get("sourceTableName")
            if not _ctx_table:
                continue
            _ctx_cols = _get_table_cols(_ctx_table)
            if _ctx_cols is None:
                _ctx_errors.append(f"  Contextualization table {_ctx_table} not accessible")
                continue
            for _sk in _ctx.get("sourceKeyRefBindings", []):
                if _sk["sourceColumnName"] not in _ctx_cols:
                    _ctx_errors.append(f"  {_ctx_table}: sourceKey '{_sk['sourceColumnName']}' not in columns")
            for _tk in _ctx.get("targetKeyRefBindings", []):
                if _tk["sourceColumnName"] not in _ctx_cols:
                    _ctx_errors.append(f"  {_ctx_table}: targetKey '{_tk['sourceColumnName']}' not in columns")
        except Exception:
            pass

# Print validation results
if _val_errors:
    print(f"  *** BINDING VALIDATION ERRORS ({len(_val_errors)}) ***")
    for _ve in _val_errors:
        print(f"    {_ve}")
if _ctx_errors:
    print(f"  *** CONTEXTUALIZATION ERRORS ({len(_ctx_errors)}) ***")
    for _ce in _ctx_errors:
        print(f"    {_ce}")
if not _val_errors and not _ctx_errors:
    fixes_msg = f" ({len(_val_fixes)} auto-fixed)" if _val_fixes else ""
    print(f"  All bindings and contextualizations validated OK{fixes_msg}")

# Check if ontology already exists
ont_id = None
r = requests.get(ONT_API, headers=headers)
if r.status_code == 200:
    for o in r.json().get("value", []):
        if o["displayName"] == ONTOLOGY_NAME:
            ont_id = o["id"]
            print(f"  Already exists (ID: {ont_id}) -- will update definition")
            break

# Fallback: try items endpoint
if not ont_id and r.status_code != 200:
    r2 = requests.get(f"{API}/workspaces/{workspace_id}/items?type=Ontology", headers=headers)
    if r2.status_code == 200:
        for o in r2.json().get("value", []):
            if o["displayName"] == ONTOLOGY_NAME:
                ont_id = o["id"]
                break

if not ont_id:
    # Create new ontology
    create_body = {
        "displayName": ONTOLOGY_NAME,
        "description": f"Healthcare Ontology -- {et_count} entity types, {rt_count} relationships, bound to lh_gold_curated",
    }
    r = requests.post(ONT_API, headers=headers, json=create_body)
    if r.status_code in (200, 201):
        ont_id = r.json().get("id")
        print(f"  Created: {ont_id}")
    elif r.status_code == 202:
        wait_lro(r, "create ontology")
        time.sleep(3)
        r2 = requests.get(ONT_API, headers=headers)
        for o in r2.json().get("value", []):
            if o["displayName"] == ONTOLOGY_NAME:
                ont_id = o["id"]
                break
        print(f"  Created (async): {ont_id}")
    elif "AlreadyInUse" in (r.text or ""):
        r2 = requests.get(ONT_API, headers=headers)
        for o in r2.json().get("value", []):
            if o["displayName"] == ONTOLOGY_NAME:
                ont_id = o["id"]
                break
        print(f"  Found existing: {ont_id}")
    else:
        print(f"  [FAIL] Create ontology: HTTP {r.status_code} {r.text[:300]}")

ont_result = "[FAIL]"
if ont_id:
    # Push definition
    update_url = f"{ONT_API}/{ont_id}/updateDefinition"
    r = requests.post(update_url, headers=headers, json={"definition": {"parts": parts}})
    if r.status_code in (200, 201):
        print(f"  [OK] Definition pushed")
        ont_result = "[OK]"
    elif r.status_code == 202:
        ok = wait_lro(r, "updateDefinition")
        ont_result = "[OK]" if ok else "[FAIL]"
        print(f"  {'[OK]' if ok else '[FAIL]'} Definition push {'succeeded' if ok else 'failed'}")
    else:
        print(f"  [FAIL] updateDefinition: HTTP {r.status_code} {r.text[:300]}")

# -- Ontology Summary --
print()
print("=" * 60)
print(f"  ONTOLOGY: {ONTOLOGY_NAME:<38} {ont_result}")
print("=" * 60)

# -- Post-deploy: Deploy graph model via standalone notebook ----------
if ont_result == "[OK]":
    print()
    print("Step 2: Deploy Graph Model")
    print("-" * 40)

    # Ensure NB_Deploy_Graph_Model exists in the workspace
    token = notebookutils.credentials.getToken("pbi")
    headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
    _items_r = requests.get(f"{API}/workspaces/{workspace_id}/items?type=Notebook", headers=headers)
    _nb_id = None
    if _items_r.status_code == 200:
        for _it in _items_r.json().get("value", []):
            if _it["displayName"] == "NB_Deploy_Graph_Model":
                _nb_id = _it["id"]
                break

    if not _nb_id:
        print("  NB_Deploy_Graph_Model not in workspace -- creating from GitHub...")
        # Download the .py source
        _py_url = f"https://raw.githubusercontent.com/{GITHUB_OWNER}/{GITHUB_REPO}/{GITHUB_BRANCH}/workspace/NB_Deploy_Graph_Model.Notebook/notebook-content.py"
        _py_r = requests.get(_py_url)
        if _py_r.status_code != 200:
            print(f"  [FAIL] Could not download notebook source: HTTP {_py_r.status_code}")
        else:
            # Convert .py -> ipynb (inline converter, self-contained)
            def _py_to_ipynb_mini(py_text):
                lines = py_text.split("\n")
                cells = []
                i = 0
                while i < len(lines) and lines[i].strip() in ("", "# Fabric notebook source"):
                    i += 1
                while i < len(lines):
                    if lines[i].startswith("# METADATA **"):
                        i += 1
                        while i < len(lines) and lines[i].strip() == "":
                            i += 1
                        if i >= len(lines):
                            break
                        if lines[i].startswith("# CELL **"):
                            ct = "code"
                        elif lines[i].startswith("# MARKDOWN **"):
                            ct = "markdown"
                        else:
                            i += 1; continue
                        i += 1
                        if i < len(lines) and lines[i].strip() == "":
                            i += 1
                        cl = []
                        while i < len(lines) and not lines[i].startswith("# METADATA **"):
                            cl.append(lines[i]); i += 1
                        while cl and cl[-1].strip() == "":
                            cl.pop()
                        if ct == "markdown":
                            cl = [(l[2:] if l.startswith("# ") else ("" if l == "#" else l)) for l in cl]
                        if cl:
                            src = [l + "\n" for l in cl[:-1]] + [cl[-1]]
                            c = {"cell_type": ct, "metadata": {}, "source": src}
                            if ct == "code":
                                c["outputs"] = []; c["execution_count"] = None
                            cells.append(c)
                    else:
                        i += 1
                nb = {"nbformat": 4, "nbformat_minor": 5,
                      "metadata": {"kernel_info": {"name": "synapse_pyspark"},
                                   "kernelspec": {"name": "synapse_pyspark", "display_name": "Synapse PySpark"},
                                   "language_info": {"name": "python"}},
                      "cells": cells}
                return json.dumps(nb, indent=1, ensure_ascii=False)

            _ipynb_str = _py_to_ipynb_mini(_py_r.text)
            _ipynb_b64 = base64.b64encode(_ipynb_str.encode("utf-8")).decode("utf-8")

            # Create notebook item
            _create_body = {
                "displayName": "NB_Deploy_Graph_Model",
                "type": "Notebook",
                "definition": {
                    "format": "ipynb",
                    "parts": [
                        {"path": "notebook-content.ipynb", "payload": _ipynb_b64, "payloadType": "InlineBase64"}
                    ]
                }
            }
            _cr = requests.post(f"{API}/workspaces/{workspace_id}/items", headers=headers, json=_create_body)
            if _cr.status_code in (200, 201):
                _nb_id = _cr.json().get("id")
                print(f"  Created: {_nb_id}")
            elif _cr.status_code == 202:
                # LRO
                _loc = _cr.headers.get("Location", "")
                if _loc:
                    for _li in range(30):
                        time.sleep(3)
                        _lr = requests.get(_loc, headers=headers)
                        if _lr.status_code == 200:
                            _lrb = _lr.json()
                            if _lrb.get("status", "").lower() in ("succeeded", "completed"):
                                break
                            elif _lrb.get("status", "").lower() in ("failed", "cancelled"):
                                break
                # Re-fetch to get ID
                _items_r2 = requests.get(f"{API}/workspaces/{workspace_id}/items?type=Notebook", headers=headers)
                if _items_r2.status_code == 200:
                    for _it in _items_r2.json().get("value", []):
                        if _it["displayName"] == "NB_Deploy_Graph_Model":
                            _nb_id = _it["id"]
                            break
                if _nb_id:
                    print(f"  Created (async): {_nb_id}")

                    # Push definition explicitly (Create Item + 202 may not apply inline def)
                    _upd_body = {"definition": {"format": "ipynb", "parts": [
                        {"path": "notebook-content.ipynb", "payload": _ipynb_b64, "payloadType": "InlineBase64"}
                    ]}}
                    _ur = requests.post(f"{API}/workspaces/{workspace_id}/items/{_nb_id}/updateDefinition",
                                        headers=headers, json=_upd_body)
                    if _ur.status_code == 202:
                        _uloc = _ur.headers.get("Location", "")
                        if _uloc:
                            for _ui in range(30):
                                time.sleep(2)
                                _ulr = requests.get(_uloc, headers=headers)
                                if _ulr.status_code == 200:
                                    break
                    print(f"  Definition pushed")
            else:
                print(f"  [FAIL] Create notebook: HTTP {_cr.status_code} {_cr.text[:300]}")

            if _nb_id:
                # Brief pause for Fabric to index the notebook
                print("  Waiting 15s for Fabric to index the notebook...")
                time.sleep(15)

    if _nb_id:
        print(f"  Running NB_Deploy_Graph_Model notebook...")
        print(f"  (validates bindings, builds graph definition, deploys,")
        print(f"   waits for data load, runs smoke tests)")
        print()
        try:
            notebookutils.notebook.run("NB_Deploy_Graph_Model", 600, {"useRootDefaultLakehouse": True})
            graph_st = "[OK]"
            print()
            print(f"  [OK] NB_Deploy_Graph_Model completed successfully")
        except Exception as _gm_err:
            graph_st = "[FAIL]"
            _gm_err_str = str(_gm_err)
            print()
            if "404" in _gm_err_str or "not found" in _gm_err_str.lower():
                print(f"  [FAIL] Fabric cannot find NB_Deploy_Graph_Model (may need more time to index)")
                print(f"         Wait 1-2 minutes, then run NB_Deploy_Graph_Model manually from the workspace")
            else:
                print(f"  [FAIL] NB_Deploy_Graph_Model failed: {_gm_err_str[:500]}")
                print(f"         Open NB_Deploy_Graph_Model manually for detailed diagnostics")
    else:
        graph_st = "[FAIL]"
        print(f"  [FAIL] Could not create NB_Deploy_Graph_Model in workspace")
        print(f"         Deploy it manually from the workspace folder, then run it")

    print()
    print("=" * 60)
    print(f"  ONTOLOGY:    {ONTOLOGY_NAME:<36} {ont_result}")
    print(f"  GRAPH:       NB_Deploy_Graph_Model{' ':>16} {graph_st}")
    print("=" * 60)


In [ ]:
# ============================================================================
# CELL 7b -- Deploy Real-Time Intelligence (RTI)
# ============================================================================
# Eventhouse + KQL Database are already deployed as Git artifacts (Stage 2).
# This cell runs post-deploy wiring and scoring notebooks:
#   1. NB_RTI_Setup_Eventhouse  -- Discovers artifacts, creates KQL tables, discovers Kusto URI
#   2. NB_RTI_Event_Simulator   -- Generates initial events (batch mode) + pushes to KQL
#   3. NB_RTI_Fraud_Detection   -- Scores claims for fraud -> Delta + KQL
#   4. NB_RTI_Care_Gap_Alerts   -- Point-of-care gap closure alerts -> Delta + KQL
#   5. NB_RTI_HighCost_Trajectory -- High-cost trajectory analysis -> Delta + KQL
# ============================================================================
# Ingestion: Direct Kusto (azure-kusto-ingest, pre-installed in Fabric)
# No Eventstream or connection strings needed -- zero config for users.
# ============================================================================

if DEPLOY_RTI:
    print("=" * 60)
    print("  REAL-TIME INTELLIGENCE DEPLOYMENT")
    print("=" * 60)

    # -- Attach lh_gold_curated to RTI notebooks -------------------------
    # Notebooks run via notebookutils.notebook.run() do not inherit the
    # caller's lakehouse context.  The child notebook MUST either have a matching default
    # lakehouse in its metadata, otherwise Fabric blocks ALL Spark SQL
    # with "No default context found" or "Cannot reference a Notebook that attaching to a different default lakehouse".
    # We patch each notebook's ipynb definition here before running them.
    # ---------------------------------------------------------------------
    import requests, base64, time as _time

    _token = notebookutils.credentials.getToken("pbi")
    _hdrs = {"Authorization": f"Bearer {_token}", "Content-Type": "application/json"}
    _api = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}"

    # Get all workspace items
    _items_resp = requests.get(f"{_api}/items", headers=_hdrs)
    _name_to_id = {(it["type"], it["displayName"]): it["id"] for it in _items_resp.json().get("value", [])}
    _lh_id = _name_to_id.get(("Lakehouse", "lh_gold_curated"), "")

    _lh_deps = {
        "lakehouse": {
            "default_lakehouse": _lh_id,
            "default_lakehouse_name": "lh_gold_curated",
            "default_lakehouse_workspace_id": workspace_id,
            "known_lakehouses": [
                {"id": _lh_id, "displayName": "lh_gold_curated", "isDefault": True}
            ],
        }
    }

    _needs_lh = ["NB_RTI_Setup_Eventhouse", "NB_RTI_Event_Simulator", "NB_RTI_Fraud_Detection",
                 "NB_RTI_Care_Gap_Alerts", "NB_RTI_HighCost_Trajectory"]

    def _get_nb_definition(nb_id, hdrs, api_base):
        """Get notebook definition in ipynb format, handling LRO."""
        # Use notebook-specific endpoint with explicit ipynb format
        url = f"{api_base}/notebooks/{nb_id}/getDefinition?format=ipynb"
        r = requests.post(url, headers=hdrs)
        print(f"      getDefinition: HTTP {r.status_code}")

        if r.status_code == 200:
            return r.json()

        if r.status_code != 202:
            print(f"      Unexpected status: {r.text[:200]}")
            return None

        # LRO - poll operation status
        loc = r.headers.get("Location", "")
        retry_secs = int(r.headers.get("Retry-After", "2"))
        if not loc:
            print(f"      No Location header in 202 response")
            return None

        print(f"      LRO polling ({retry_secs}s interval)...")
        for attempt in range(60):
            _time.sleep(retry_secs)
            poll_r = requests.get(loc, headers=hdrs)
            if poll_r.status_code == 202:
                continue
            if poll_r.status_code != 200:
                print(f"      LRO poll: HTTP {poll_r.status_code}")
                return None

            body = poll_r.json()
            status = body.get("status", "")
            if status == "Running" or status == "NotStarted":
                continue
            if status != "Succeeded":
                print(f"      LRO status: {status}")
                return None

            # LRO succeeded - fetch the actual result
            # Try {loc}/result FIRST (most reliable)
            result_url = f"{loc}/result"
            print(f"      LRO succeeded, fetching /result...")
            result_r = requests.get(result_url, headers=hdrs)
            if result_r.status_code == 200:
                result_body = result_r.json()
                parts = result_body.get("definition", {}).get("parts", [])
                if not parts:
                    parts = result_body.get("parts", [])
                if parts:
                    print(f"      Got {len(parts)} parts from /result")
                    return result_body

            # Try resourceLocation (sometimes points to item URL, not definition)
            res_loc = body.get("resourceLocation", "")
            if res_loc and res_loc != result_url:
                print(f"      Trying resourceLocation...")
                res_r = requests.get(res_loc, headers=hdrs)
                if res_r.status_code == 200:
                    res_body = res_r.json()
                    parts = res_body.get("definition", {}).get("parts", [])
                    if not parts:
                        parts = res_body.get("parts", [])
                    if parts:
                        print(f"      Got {len(parts)} parts from resourceLocation")
                        return res_body

            # Try the poll body itself (some APIs embed result in final status)
            parts = body.get("definition", {}).get("parts", [])
            if not parts:
                parts = body.get("parts", [])
            if parts:
                print(f"      Got {len(parts)} parts from poll body")
                return body

            print(f"      WARNING: LRO succeeded but no parts found in any endpoint")
            return None

        print(f"      LRO timed out after 60 polls")
        return None

    def _convert_pip_cells(nb_json):
        """Convert %pip/%conda cells to subprocess + module reload equivalents.

        Fabric blocks %pip/%conda magic in child notebooks called via
        notebookutils.notebook.run(). Converting to subprocess calls
        achieves the same package installation and works in child notebooks.
        After install, purges azure.* from sys.modules so the new
        versions load on next import (subprocess doesn't restart kernel).
        Returns the number of cells converted.
        """
        if "cells" not in nb_json:
            return 0
        converted = 0
        for cell in nb_json["cells"]:
            src_text = "".join(cell.get("source", []))
            stripped = src_text.strip()
            if not stripped:
                continue
            lines = stripped.split("\n")
            if not all(
                line.strip().startswith(("%pip ", "%conda "))
                or line.strip() == ""
                for line in lines
            ):
                continue
            # Build subprocess equivalent
            new_lines = ["import subprocess, sys\n"]
            for line in lines:
                line = line.strip()
                if line.startswith("%pip install "):
                    pkgs = line[len("%pip install "):].split()
                    pkg_str = ", ".join(f'"{p}"' for p in pkgs)
                    new_lines.append(
                        f'subprocess.check_call([sys.executable, "-m", "pip", "install", {pkg_str}])\n'
                    )
                elif line.startswith("%conda install "):
                    pkgs = line[len("%conda install "):].split()
                    pkg_str = ", ".join(f'"{p}"' for p in pkgs)
                    new_lines.append(
                        f'subprocess.check_call(["conda", "install", "-y", {pkg_str}])\n'
                    )
            # Purge cached azure.* modules so re-import picks up new versions
            new_lines.append("# Purge old azure modules so fresh versions load\n")
            new_lines.append("for _mod in sorted(sys.modules):\n")
            new_lines.append("    if _mod.startswith('azure'):\n")
            new_lines.append("        del sys.modules[_mod]\n")
            cell["source"] = new_lines
            cell["cell_type"] = "code"
            converted += 1
        return converted

    if _lh_id:
        print(f"\n  Attaching lh_gold_curated ({_lh_id[:8]}...) to RTI notebooks...")
        for _nb_name in _needs_lh:
            _nb_id = _name_to_id.get(("Notebook", _nb_name))
            if not _nb_id:
                print(f"    {_nb_name}: not found in workspace, skipping")
                continue

            print(f"    {_nb_name}:")
            result = _get_nb_definition(_nb_id, _hdrs, _api)

            if result is None:
                print(f"      SKIP - could not get definition")
                continue

            # Extract parts from response (handle both nesting patterns)
            _defn = result.get("definition", result)
            _parts = _defn.get("parts", [])

            if not _parts:
                print(f"      SKIP - 0 parts in response")
                print(f"      Response keys: {list(result.keys())}")
                if "definition" in result:
                    print(f"      definition keys: {list(result['definition'].keys())}")
                continue

            print(f"      Parts: {[p.get('path','?') for p in _parts]}")

            # Find and patch the notebook content part
            _patched = False
            for _part in _parts:
                _path = _part.get("path", "")
                # Match ipynb content parts (various naming patterns)
                if _path.endswith(".ipynb") or "content" in _path.lower() or _path == "artifact.content.ipynb":
                    try:
                        _raw = base64.b64decode(_part["payload"]).decode("utf-8")
                        _nb_json = json.loads(_raw)
                        # Patch metadata with lakehouse dependency
                        _meta = _nb_json.setdefault("metadata", {})
                        _meta["trident"] = _lh_deps
                        _meta["dependencies"] = _lh_deps
                        # Strip %pip/%conda cells (blocked in notebook.run())
                        _converted = _convert_pip_cells(_nb_json)
                        _part["payload"] = base64.b64encode(
                            json.dumps(_nb_json).encode("utf-8")
                        ).decode("utf-8")
                        _patched = True
                        print(f"      Patched lakehouse into {_path}")
                        if _converted:
                            print(f"      Converted {_converted} %pip/%conda cell(s) to subprocess")
                    except json.JSONDecodeError:
                        # Not JSON (probably .py format) -- skip this part
                        print(f"      {_path} is not JSON, trying next part...")
                        continue
                    except Exception as _ex:
                        print(f"      Failed to patch {_path}: {_ex}")
                        continue
                    break

            if not _patched:
                print(f"      Could not patch any part (no ipynb content found)")
                # As a last resort, try to find ANY JSON part we can decode
                for _part in _parts:
                    _path = _part.get("path", "")
                    if _path == ".platform":
                        continue
                    try:
                        _raw = base64.b64decode(_part["payload"]).decode("utf-8")
                        _nb_json = json.loads(_raw)
                        if "cells" in _nb_json or "nbformat" in _nb_json:
                            _meta = _nb_json.setdefault("metadata", {})
                            _meta["trident"] = _lh_deps
                            _meta["dependencies"] = _lh_deps
                            _converted = _convert_pip_cells(_nb_json)
                            _part["payload"] = base64.b64encode(
                                json.dumps(_nb_json).encode("utf-8")
                            ).decode("utf-8")
                            _patched = True
                            print(f"      Patched lakehouse into {_path} (by content detection)")
                            if _converted:
                                print(f"      Converted {_converted} %pip/%conda cell(s) to subprocess")
                            break
                    except Exception:
                        continue

            if not _patched:
                print(f"      SKIP - no patchable content part found")
                continue

            # Push updated definition back
            _update_body = {"definition": {"format": "ipynb", "parts": _parts}}
            _ur = requests.post(
                f"{_api}/notebooks/{_nb_id}/updateDefinition?updateMetadata=true",
                headers=_hdrs,
                json=_update_body,
            )
            if _ur.status_code == 200:
                print(f"      Lakehouse attached (200)")
            elif _ur.status_code == 202:
                _uloc = _ur.headers.get("Location", "")
                _update_ok = False
                if _uloc:
                    for _poll_i in range(30):
                        _time.sleep(2)
                        _pr = requests.get(_uloc, headers=_hdrs)
                        if _pr.status_code == 200:
                            _pstatus = _pr.json().get("status", "")
                            if _pstatus == "Succeeded":
                                _update_ok = True
                                break
                            elif _pstatus in ("Failed", "Cancelled"):
                                _perr = _pr.json().get("error", {})
                                print(f"      updateDefinition {_pstatus}: "
                                      f"{_perr.get('message', '')[:200]}")
                                break
                        elif _pr.status_code != 202:
                            # Non-standard status, assume done
                            _update_ok = True
                            break
                else:
                    _update_ok = True
                if _update_ok:
                    print(f"      Lakehouse attached (202->done)")
                else:
                    print(f"      updateDefinition may have failed "
                          f"-- notebook may not pick up lakehouse")
            else:
                print(f"      updateDefinition failed: HTTP {_ur.status_code} {_ur.text[:300]}")
    else:
        print("  WARNING: lh_gold_curated not found in workspace -- cannot attach lakehouse")

    # -- Run RTI notebooks ------------------------------------------------
    # Brief delay to let Fabric propagate the updateDefinition changes
    # (lakehouse metadata + %pip cell removal) before running child notebooks.
    print("\n  Waiting 15s for notebook metadata propagation...")
    _time.sleep(15)

    rti_notebooks = [
        "NB_RTI_Setup_Eventhouse",
        "NB_RTI_Event_Simulator",
        "NB_RTI_Fraud_Detection",
        "NB_RTI_Care_Gap_Alerts",
        "NB_RTI_HighCost_Trajectory",
    ]

    for nb_name in rti_notebooks:
        print(f"\n  Running {nb_name}...")
        try:
            notebookutils.notebook.run(nb_name, 1200, {"useRootDefaultLakehouse": True})
            print(f"  -> {nb_name}: OK")
        except Exception as e:
            print(f"  -> {nb_name}: FAILED -- {e}")
            print(f"    You can run this notebook manually from the workspace.")

    print("\n" + "=" * 60)
    print("  RTI DEPLOYMENT COMPLETE")
    print("=" * 60)
    print("  Delta tables in lh_gold_curated:")
    print("    - rti_claims_events, rti_adt_events, rti_rx_events")
    print("    - rti_fraud_scores, rti_care_gap_alerts, rti_highcost_alerts")
    print()
    print("  KQL tables in Healthcare_RTI_Eventhouse:")
    print("    - claims_events, adt_events, rx_events")
    print("    - fraud_scores, care_gap_alerts, highcost_alerts")
    print()
    print("  Ingestion: Direct Kusto (zero config)")
    print("  For streaming: set MODE='stream' in NB_RTI_Event_Simulator")
    print("=" * 60)
else:
    print("Skipping RTI deployment (DEPLOY_RTI=False)")


In [ ]:
# ============================================================================
# CELL 7b -- Patch Data Agent sources with real workspace/artifact IDs
# ============================================================================
# fabric-launcher deploys the DataAgent from the repo's exported definition,
# but datasource.json files contain placeholder IDs:
#   artifactId:  a1000001-0001-0001-0001-000000000004
#   workspaceId: 00000000-0000-0000-0000-000000000000
#
# This cell patches those placeholders with the REAL lakehouse and semantic
# model IDs discovered at runtime, then pushes the updated definition.
# Also adds dataSourceInstructions to the semantic model datasource.
# ============================================================================

import requests, json, base64, time

print("Patching Data Agent sources with real IDs...\n")

token = notebookutils.credentials.getToken("pbi")
headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
api_base = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}"

# --- 1. Find the DataAgent and data sources in workspace ---
resp = requests.get(f"{api_base}/items", headers=headers)
items = resp.json().get("value", []) if resp.status_code == 200 else []

agent_id = None
lh_gold_id = None
sm_id = None

for item in items:
    if item["type"] == "DataAgent" and item["displayName"] == "HealthcareHLSAgent":
        agent_id = item["id"]
    elif item["type"] == "Lakehouse" and item["displayName"] == "lh_gold_curated":
        lh_gold_id = item["id"]
    elif item["type"] == "SemanticModel" and item["displayName"] == "HealthcareDemoHLS":
        sm_id = item["id"]

if not agent_id:
    print("[SKIP] HealthcareHLSAgent not found in workspace")
elif not lh_gold_id:
    print("[SKIP] lh_gold_curated lakehouse not found")
else:
    print(f"  Agent:          {agent_id}")
    print(f"  Lakehouse:      {lh_gold_id}")
    print(f"  Semantic Model: {sm_id or '(not found -- will skip SM source)'}")

    # --- 2. Get current agent definition ---
    get_def_url = f"{api_base}/DataAgents/{agent_id}/getDefinition"
    r = requests.post(get_def_url, headers=headers)
    print(f"  getDefinition: HTTP {r.status_code}")

    parts = None
    if r.status_code == 200:
        parts = r.json().get("definition", {}).get("parts", [])
    elif r.status_code == 202:
        # LRO -- poll for completion
        op_url = r.headers.get("Location", "")
        retry_after = int(r.headers.get("Retry-After", 5))
        for _ in range(60):
            time.sleep(retry_after)
            op_r = requests.get(op_url, headers=headers)
            if op_r.status_code == 200:
                body = op_r.json()
                if body.get("status") == "Succeeded":
                    # Try resourceLocation first, then /result
                    res_loc = body.get("resourceLocation", "")
                    for result_url in [f"{op_url}/result", res_loc]:
                        if not result_url:
                            continue
                        rr = requests.get(result_url, headers=headers)
                        if rr.status_code == 200:
                            p = rr.json().get("definition", {}).get("parts", [])
                            if p:
                                parts = p
                                break
                    if not parts:
                        # Body itself might have parts
                        parts = body.get("definition", {}).get("parts", [])
                    break
                elif body.get("status") in ("Failed", "Cancelled"):
                    print(f"  [FAIL] getDefinition LRO: {body.get('status')}")
                    break
    else:
        print(f"  [FAIL] getDefinition: {r.status_code} {r.text[:200]}")

    if not parts:
        print("  [FAIL] Could not get agent definition parts")
    else:
        print(f"  Definition parts: {len(parts)}")

        # --- 3. Mapping: datasource displayName -> real IDs ---
        ds_type_map = {
            "lakehouse_tables": {"artifactId": lh_gold_id, "workspaceId": workspace_id},
        }
        if sm_id:
            ds_type_map["semantic_model"] = {"artifactId": sm_id, "workspaceId": workspace_id}

        # SM instructions to inject (currently null in the exported definition)
        sm_instructions = (
            "Use this semantic model for pre-built DAX measures and aggregations. "
            "It contains 26 measures across encounters, claims, prescriptions, and adherence tables. "
            "Use DAX measures when the user asks for KPIs, rates, or summary metrics like "
            "'denial rate', 'readmission rate', 'adherent rate', 'total charges', 'avg length of stay'. "
            "For row-level detail, patient lookups, or multi-table JOINs, use the lakehouse SQL source instead."
        )
        sm_description = (
            "HealthcareDemoHLS Direct Lake semantic model with 26 pre-built DAX measures "
            "for healthcare KPIs: denial rates, readmission rates, medication adherence, "
            "charges, encounter volumes, and cost trends."
        )

        # --- 4. Patch datasource.json parts ---
        patched_count = 0
        patched_parts = []

        for part in parts:
            part_path = part.get("path", "")
            payload = part.get("payload", "")
            payload_type = part.get("payloadType", "InlineBase64")

            if "datasource.json" in part_path and payload_type == "InlineBase64" and payload:
                try:
                    content = base64.b64decode(payload).decode("utf-8")
                    ds = json.loads(content)
                    ds_name = ds.get("displayName", "")
                    ds_type = ds.get("type", "")
                    old_artifact = ds.get("artifactId", "")
                    old_workspace = ds.get("workspaceId", "")

                    if ds_type in ds_type_map:
                        new_ids = ds_type_map[ds_type]
                        new_artifact = new_ids["artifactId"]
                        new_workspace = new_ids["workspaceId"]

                        if old_artifact != new_artifact:
                            print(f"    Patch {ds_name} artifactId: {old_artifact[:20]}... -> {new_artifact[:20]}...")
                            ds["artifactId"] = new_artifact
                        if old_workspace != new_workspace:
                            print(f"    Patch {ds_name} workspaceId: {old_workspace[:20]}... -> {new_workspace[:20]}...")
                            ds["workspaceId"] = new_workspace

                        # Add SM instructions if this is the semantic model source
                        if ds_type == "semantic_model":
                            if not ds.get("dataSourceInstructions"):
                                ds["dataSourceInstructions"] = sm_instructions
                                print(f"    Added dataSourceInstructions to {ds_name}")
                            if not ds.get("userDescription"):
                                ds["userDescription"] = sm_description
                                print(f"    Added userDescription to {ds_name}")

                        content = json.dumps(ds, indent=2, ensure_ascii=False)
                        payload = base64.b64encode(content.encode("utf-8")).decode("utf-8")
                        patched_count += 1
                    else:
                        print(f"    [SKIP] {ds_name} ({ds_type}) -- no mapping available")
                except Exception as e:
                    print(f"    [WARN] Could not patch {part_path}: {e}")

            patched_parts.append({
                "path": part_path,
                "payload": payload,
                "payloadType": payload_type,
            })

        print(f"  Patched {patched_count} datasource files")

        # --- 5. Push updated definition ---
        if patched_count > 0:
            update_url = f"{api_base}/DataAgents/{agent_id}/updateDefinition"
            update_body = {"definition": {"parts": patched_parts}}
            r = requests.post(update_url, headers=headers, json=update_body)
            print(f"  updateDefinition: HTTP {r.status_code}")

            if r.status_code == 200:
                print("  [OK] Agent definition updated successfully")
            elif r.status_code == 202:
                # LRO -- wait for completion
                op_url = r.headers.get("Location", "")
                retry_after = int(r.headers.get("Retry-After", 5))
                for _ in range(60):
                    time.sleep(retry_after)
                    op_r = requests.get(op_url, headers=headers)
                    if op_r.status_code == 200:
                        body = op_r.json()
                        if body.get("status") == "Succeeded":
                            print("  [OK] Agent definition updated (LRO succeeded)")
                            break
                        elif body.get("status") in ("Failed", "Cancelled"):
                            err = body.get("error", {}).get("message", "")
                            print(f"  [FAIL] updateDefinition LRO: {body.get('status')} -- {err}")
                            break
                else:
                    print("  [WARN] updateDefinition LRO timed out")
            else:
                print(f"  [FAIL] updateDefinition: {r.text[:300]}")
        else:
            print("  [SKIP] No datasource files needed patching")

print("\nData Agent source patching complete.")


In [ ]:
# ============================================================================
# CELL 7c -- Deploy Real-Time Dashboard (KQL Dashboard)
# ============================================================================
# Creates a Real-Time Dashboard connected to the Eventhouse KQL database.
# 4 pages, 20 KQL queries, 30-second auto-refresh:
#   Page 1: Real-Time Operations -- live KPIs, event volume, fraud distribution
#   Page 2: Care Quality & Gaps -- care gap alerts, overdue measures, high-cost
#   Page 3: Fraud Detection & Claims -- fraud trends, watch lists, claim types
#   Page 4: SLA & Data Freshness -- table freshness, ingestion rates, row counts
# ============================================================================

import json, base64, time, requests

DASHBOARD_NAME = "Healthcare RTI Dashboard"
FABRIC_API = "https://api.fabric.microsoft.com/v1"

token = notebookutils.credentials.getToken("pbi")
headers = {"Authorization": f"Bearer {token}"}
workspace_id = notebookutils.runtime.context.get("currentWorkspaceId", "")

print(f"Workspace: {workspace_id}")

# -- Step 1: Find Eventhouse and get query URI ----------------------------
print("\n[Step 1] Finding Eventhouse...")
eh_id = None
query_uri = None
kql_db_name = None

resp = requests.get(f"{FABRIC_API}/workspaces/{workspace_id}/items?type=Eventhouse", headers=headers)
if resp.status_code == 200:
    for item in resp.json().get("value", []):
        eh_id = item["id"]
        print(f"  Eventhouse: {item['displayName']} ({eh_id})")
        break

if eh_id:
    resp = requests.get(f"{FABRIC_API}/workspaces/{workspace_id}/eventhouses/{eh_id}", headers=headers)
    if resp.status_code == 200:
        props = resp.json().get("properties", {})
        query_uri = props.get("queryServiceUri") or props.get("uri")
        print(f"  Query URI: {query_uri}")

# Find KQL database name and ID
kql_db_id = None
resp = requests.get(f"{FABRIC_API}/workspaces/{workspace_id}/items?type=KQLDatabase", headers=headers)
if resp.status_code == 200:
    for item in resp.json().get("value", []):
        kql_db_name = item["displayName"]
        kql_db_id = item["id"]
        print(f"  KQL Database: {kql_db_name} ({kql_db_id})")
        break

if not eh_id or not query_uri:
    print("[SKIP] No Eventhouse found -- skipping dashboard deployment")
else:
    # -- Step 2: Load dashboard template from GitHub --------------------------
    print("\n[Step 2] Loading dashboard template...")
    
    # Try local filesystem first (fabric-launcher extracted repo)
    dashboard_json = None
    local_paths = [
        "/lakehouse/default/Files/src/rti_dashboard/healthcare_rti_dashboard.json",
    ]
    # Also check subdirectory pattern (extracted ZIP)
    import os
    src_base = "/lakehouse/default/Files/src"
    if os.path.isdir(src_base):
        for d in os.listdir(src_base):
            candidate = os.path.join(src_base, d, "rti_dashboard", "healthcare_rti_dashboard.json")
            local_paths.append(candidate)
    
    for lpath in local_paths:
        if os.path.exists(lpath):
            with open(lpath, "r", encoding="utf-8") as f:
                dashboard_json = f.read()
            print(f"  Loaded from: {lpath}")
            break
    
    if not dashboard_json:
        # Fallback: download from GitHub
        gh_url = f"https://raw.githubusercontent.com/{GITHUB_OWNER}/{GITHUB_REPO}/{GITHUB_BRANCH}/rti_dashboard/healthcare_rti_dashboard.json"
        print(f"  Downloading from GitHub: {gh_url}")
        resp = requests.get(gh_url)
        if resp.status_code == 200:
            dashboard_json = resp.text
            print(f"  Downloaded ({len(dashboard_json)} bytes)")
        else:
            print(f"  [FAIL] GitHub download failed: HTTP {resp.status_code}")
    
    if dashboard_json:
        # -- Step 3: Patch placeholders ----------------------------------------
        print("\n[Step 3] Patching Eventhouse connection...")
        dashboard_json = dashboard_json.replace("__EVENTHOUSE_QUERY_URI__", query_uri)
        dashboard_json = dashboard_json.replace("__EVENTHOUSE_ID__", eh_id)
        if kql_db_id:
            dashboard_json = dashboard_json.replace("__KQL_DB_ID__", kql_db_id)
        if kql_db_name:
            dashboard_json = dashboard_json.replace("__KQL_DB_NAME__", kql_db_name)
        
        # Validate
        dash_def = json.loads(dashboard_json)
        n_pages = len(dash_def.get("pages", []))
        n_queries = len(dash_def.get("queries", []))
        total_tiles = sum(len(p.get("tiles", [])) for p in dash_def.get("pages", []))
        print(f"  Dashboard: {n_pages} pages, {total_tiles} tiles, {n_queries} queries")
        
        # -- Step 4: Create or update dashboard --------------------------------
        # -- Step 4: Deploy dashboard -----------------------------------------
        print("\n[Step 4] Deploying dashboard...")
        
        payload_b64 = base64.b64encode(dashboard_json.encode("utf-8")).decode("utf-8")
        parts = [{
            "path": "RealTimeDashboard.json",
            "payload": payload_b64,
            "payloadType": "InlineBase64"
        }]
        
        # Delete any existing dashboard first
        deleted = False
        for item_type in ["RealTimeDashboard", "KQLDashboard", "Dashboard"]:
            if deleted:
                break
            resp = requests.get(
                f"{FABRIC_API}/workspaces/{workspace_id}/items?type={item_type}",
                headers=headers
            )
            if resp.status_code == 200:
                for item in resp.json().get("value", []):
                    if item["displayName"] == DASHBOARD_NAME:
                        old_id = item["id"]
                        print(f"  Found existing {item_type} ({old_id}) -- deleting...")
                        del_resp = requests.delete(
                            f"{FABRIC_API}/workspaces/{workspace_id}/items/{old_id}",
                            headers=headers
                        )
                        if del_resp.status_code in (200, 204):
                            print(f"  [OK] Deleted old dashboard")
                            deleted = True
                        else:
                            print(f"  [WARN] Delete HTTP {del_resp.status_code}")
                        break
        if deleted:
            print("  Waiting 20s for delete to propagate...")
            time.sleep(20)
        
        # Create dashboard with definition inline (not empty + updateDefinition)
        # Try RealTimeDashboard first, fall back to KQLDashboard
        dash_id = None
        for item_type in ["RealTimeDashboard", "KQLDashboard"]:
            url = f"{FABRIC_API}/workspaces/{workspace_id}/items"
            body = {
                "displayName": DASHBOARD_NAME,
                "type": item_type,
                "definition": {"parts": parts},
            }
            print(f"  Creating {item_type} with inline definition...")
            resp = requests.post(url, headers=headers, json=body)
            
            if resp.status_code in (200, 201):
                dash_id = resp.json().get("id", "")
                print(f"  [OK] Created: {DASHBOARD_NAME} (type={item_type}, id={dash_id})")
                break
            elif resp.status_code == 202:
                op_url = resp.headers.get("Location", "")
                _ok = False
                for _ in range(60):
                    time.sleep(5)
                    op_resp = requests.get(op_url, headers=headers)
                    if op_resp.status_code == 200:
                        status = op_resp.json().get("status", "")
                        if status == "Succeeded":
                            _ok = True
                            break
                        elif status in ("Failed", "Cancelled"):
                            err = op_resp.json().get("error", {}).get("message", "")
                            print(f"  [FAIL] {item_type} create {status}: {err}")
                            break
                if _ok:
                    # Find the created item
                    list_resp = requests.get(
                        f"{FABRIC_API}/workspaces/{workspace_id}/items?type={item_type}",
                        headers=headers
                    )
                    if list_resp.status_code == 200:
                        for itm in list_resp.json().get("value", []):
                            if itm["displayName"] == DASHBOARD_NAME:
                                dash_id = itm["id"]
                                break
                    if dash_id:
                        print(f"  [OK] Created: {DASHBOARD_NAME} (type={item_type}, id={dash_id})")
                    break
                else:
                    print(f"  {item_type} failed, trying next type...")
                    continue
            elif resp.status_code == 409:
                # Already exists — find and update it
                print(f"  [INFO] {item_type} already exists, finding and updating...")
                time.sleep(10)
                list_resp = requests.get(
                    f"{FABRIC_API}/workspaces/{workspace_id}/items?type={item_type}",
                    headers=headers
                )
                if list_resp.status_code == 200:
                    for itm in list_resp.json().get("value", []):
                        if itm["displayName"] == DASHBOARD_NAME:
                            dash_id = itm["id"]
                            break
                if dash_id:
                    # Push definition via updateDefinition
                    print(f"  Updating definition on {dash_id}...")
                    ur = requests.post(
                        f"{FABRIC_API}/workspaces/{workspace_id}/items/{dash_id}/updateDefinition",
                        headers=headers, json={"definition": {"parts": parts}}
                    )
                    if ur.status_code in (200, 201):
                        print(f"  [OK] Definition updated")
                    elif ur.status_code == 202:
                        _uloc = ur.headers.get("Location", "")
                        for _ in range(30):
                            time.sleep(5)
                            _upr = requests.get(_uloc, headers=headers)
                            if _upr.status_code == 200:
                                _us = _upr.json().get("status", "")
                                if _us == "Succeeded":
                                    print(f"  [OK] Definition updated")
                                    break
                                elif _us in ("Failed", "Cancelled"):
                                    print(f"  [FAIL] updateDefinition: {_us}")
                                    break
                    else:
                        print(f"  [WARN] updateDefinition HTTP {ur.status_code}: {ur.text[:300]}")
                break
            else:
                err_text = resp.text[:300]
                # If type not supported, try next
                if any(kw in err_text.lower() for kw in ["invaliditemtype", "unsupported", "not supported"]):
                    print(f"  {item_type} not supported, trying next...")
                    continue
                print(f"  [WARN] HTTP {resp.status_code}: {err_text}")
                break
        
        if not dash_id:
            print("  [FAIL] Could not create dashboard with any item type")
        
        print("\n[DONE] RTI Dashboard deployment complete")
        print(f"  Open the dashboard in the Fabric workspace to see live KQL data")
        print(f"  Auto-refresh: 30 seconds")
    else:
        print("[FAIL] Could not load dashboard template")


In [ ]:
# ============================================================================
# CELL 8 — Organize Workspace Folders & Deployment Summary
# ============================================================================
# Creates logical folders and moves deployed items into them, then prints
# a full deployment summary.
#
# Folder structure:
#   Data/               — Lakehouses, SQLEndpoints
#   Notebooks/          — All NB_* notebooks (not this launcher)
#   Pipelines/          — DataPipeline items
#   Analytics/          — SemanticModel, DataAgent, Ontology, GraphModel
#   Real-Time Intelligence/ — Eventhouse, KQLDatabase, KQLDashboard,
#                             Eventstream, RealTimeDashboard
#
# Items not matching any rule stay at workspace root.
# Folder creation is idempotent (skips if folder already exists).
# ============================================================================

import requests, time

token = notebookutils.credentials.getToken("pbi")
headers = {
    "Authorization": f"Bearer {token}",
    "Content-Type": "application/json",
}
api_base = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}"

# ── 1. Enumerate workspace items ──────────────────────────────
resp = requests.get(f"{api_base}/items", headers=headers)
items = resp.json().get("value", []) if resp.status_code == 200 else []

# ── 2. Define folder → item type mapping ──────────────────────
FOLDER_RULES = {
    "Data": {"Lakehouse", "SQLEndpoint"},
    "Notebooks": {"Notebook"},
    "Pipelines": {"DataPipeline"},
    "Analytics": {"SemanticModel", "DataAgent", "Ontology", "GraphModel",
                  "GraphQuerySet"},
    "Real-Time Intelligence": {"Eventhouse", "KQLDatabase", "KQLDashboard",
                               "Eventstream", "RealTimeDashboard"},
}

# Reverse map: item_type → folder_name
type_to_folder = {}
for folder_name, item_types in FOLDER_RULES.items():
    for it in item_types:
        type_to_folder[it] = folder_name

# ── 3. List existing folders (avoid duplicates) ──────────────
existing_folders = {}  # displayName → folder_id
fr = requests.get(f"{api_base}/folders", headers=headers)
if fr.status_code == 200:
    for f in fr.json().get("value", []):
        existing_folders[f["displayName"]] = f["id"]

# ── 4. Create missing folders ────────────────────────────────
print("=" * 60)
print("  ORGANIZE WORKSPACE FOLDERS")
print("=" * 60)

folder_ids = {}  # folder_name → folder_id
for folder_name in FOLDER_RULES:
    if folder_name in existing_folders:
        folder_ids[folder_name] = existing_folders[folder_name]
        print(f"  [OK] Folder exists: {folder_name}")
    else:
        cr = requests.post(
            f"{api_base}/folders",
            headers=headers,
            json={"displayName": folder_name},
        )
        if cr.status_code == 201:
            fid = cr.json()["id"]
            folder_ids[folder_name] = fid
            print(f"  [OK] Created folder: {folder_name}")
        elif cr.status_code == 429:
            retry_after = int(cr.headers.get("Retry-After", 10))
            print(f"  [WAIT] Rate limited, retrying in {retry_after}s...")
            time.sleep(retry_after)
            token = notebookutils.credentials.getToken("pbi")
            headers["Authorization"] = f"Bearer {token}"
            cr2 = requests.post(
                f"{api_base}/folders",
                headers=headers,
                json={"displayName": folder_name},
            )
            if cr2.status_code == 201:
                fid = cr2.json()["id"]
                folder_ids[folder_name] = fid
                print(f"  [OK] Created folder: {folder_name}")
            else:
                print(f"  [FAIL] Create folder '{folder_name}': "
                      f"HTTP {cr2.status_code} {cr2.text[:200]}")
        else:
            err_code = ""
            try:
                err_code = cr.json().get("errorCode", "")
            except Exception:
                pass
            if "AlreadyInUse" in err_code or "DisplayNameAlreadyInUse" in err_code:
                # Race condition — re-list folders
                fr2 = requests.get(f"{api_base}/folders", headers=headers)
                for fobj in fr2.json().get("value", []):
                    if fobj["displayName"] == folder_name:
                        folder_ids[folder_name] = fobj["id"]
                        print(f"  [OK] Folder already exists: {folder_name}")
                        break
            else:
                print(f"  [FAIL] Create folder '{folder_name}': "
                      f"HTTP {cr.status_code} {cr.text[:200]}")

print(f"  Folders ready: {len(folder_ids)}/{len(FOLDER_RULES)}")

# ── 5. Move items into folders ───────────────────────────────
print()
print("  Moving items to folders...")
moved, skipped, failed = 0, 0, 0

# Exclude the launcher notebook from being moved
LAUNCHER_NAME = "Healthcare_Launcher"

for it in items:
    item_type = it.get("type", "")
    item_name = it.get("displayName", "")
    item_id = it["id"]
    target_folder = type_to_folder.get(item_type)

    # Skip items with no folder rule, or folder not created
    if not target_folder or target_folder not in folder_ids:
        skipped += 1
        continue

    # Don't move the launcher notebook itself
    if item_type == "Notebook" and item_name == LAUNCHER_NAME:
        skipped += 1
        continue

    # Skip if item is already in the correct folder
    if it.get("folderId") == folder_ids[target_folder]:
        skipped += 1
        continue

    mr = requests.post(
        f"{api_base}/items/{item_id}/move",
        headers=headers,
        json={"targetFolderId": folder_ids[target_folder]},
    )
    if mr.status_code == 200:
        moved += 1
    elif mr.status_code == 429:
        retry_after = int(mr.headers.get("Retry-After", 5))
        time.sleep(retry_after)
        token = notebookutils.credentials.getToken("pbi")
        headers["Authorization"] = f"Bearer {token}"
        mr2 = requests.post(
            f"{api_base}/items/{item_id}/move",
            headers=headers,
            json={"targetFolderId": folder_ids[target_folder]},
        )
        if mr2.status_code == 200:
            moved += 1
        else:
            print(f"    [FAIL] Move {item_type}/{item_name}: "
                  f"HTTP {mr2.status_code} {mr2.text[:150]}")
            failed += 1
    else:
        # Some child items move with parent automatically
        err_code = ""
        try:
            err_code = mr.json().get("errorCode", "")
        except Exception:
            pass
        if "CannotMoveChildOnly" in err_code:
            skipped += 1
        else:
            print(f"    [FAIL] Move {item_type}/{item_name}: "
                  f"HTTP {mr.status_code} "
                  f"{err_code or mr.text[:150]}")
            failed += 1

print(f"  Done: {moved} moved, {skipped} skipped, {failed} failed")

# ── 6. Deployment Summary ────────────────────────────────────
print()

# Re-fetch items to get updated folder assignments
token = notebookutils.credentials.getToken("pbi")
headers["Authorization"] = f"Bearer {token}"
resp = requests.get(f"{api_base}/items", headers=headers)
items = resp.json().get("value", []) if resp.status_code == 200 else []

# Reverse folder_id → folder_name lookup
fid_to_name = {v: k for k, v in folder_ids.items()}

# Group by folder, then by type
by_folder = {}
for it in items:
    fid = it.get("folderId")
    folder = fid_to_name.get(fid, "(root)")
    by_folder.setdefault(folder, {})
    t = it.get("type", "Unknown")
    by_folder[folder].setdefault(t, []).append(it["displayName"])

print("=" * 60)
print("  DEPLOYMENT COMPLETE")
print("=" * 60)
print(f"  Workspace: {workspace_id}")
print(f"  Total items: {len(items)}")
print()

# Print organized by folder
for folder_name in list(FOLDER_RULES.keys()) + ["(root)"]:
    types = by_folder.get(folder_name, {})
    if not types:
        continue
    total = sum(len(v) for v in types.values())
    print(f"  [{folder_name}] ({total} items)")
    for item_type in sorted(types.keys()):
        names = types[item_type]
        print(f"    {item_type} ({len(names)}):")
        for n in sorted(names):
            print(f"      - {n}")
    print()

# RTI table counts
print("  RTI Delta Tables (lh_gold_curated):")
try:
    for tbl in ["rti_claims_events", "rti_adt_events", "rti_rx_events",
                "rti_fraud_scores", "rti_care_gap_alerts",
                "rti_highcost_alerts"]:
        try:
            cnt = spark.table(tbl).count()
            print(f"    - {tbl}: {cnt:,} rows")
        except Exception:
            print(f"    - {tbl}: (not created yet)")
except Exception:
    print("    (could not query RTI tables)")

print()
print("=" * 60)
print("  NEXT STEPS")
print("=" * 60)
print("  1. Open the HealthcareDemoHLS semantic model -> verify tables loaded")
print("  2. Open the HealthcareHLSAgent -> ask: 'What are the top denial reasons?'")
print("  3. Verify the Ontology + Graph Model (auto-provisioned, or run NB_Deploy_Graph_Model)")
print("     -> Open Healthcare_Demo_Ontology_HLS in workspace to explore")
print("  4. For incremental loads: run PL_Healthcare_Master with "
      "load_mode=incremental")
print("     (auto-generates incremental data + runs full ETL in one shot)")
print("  5. RTI Dashboard: Open Healthcare_RTI_Dashboard for live KQL visuals")
print("     - NB_RTI_Fraud_Detection: View rti_fraud_scores for flagged claims")
print("     - NB_RTI_Care_Gap_Alerts: View rti_care_gap_alerts for gap closures")
print("     - NB_RTI_HighCost_Trajectory: View rti_highcost_alerts for "
      "cost trends")
print("  6. For live streaming: set MODE='stream' in NB_RTI_Event_Simulator")
print("     and configure the Eventstream Custom App connection string")
print("=" * 60)